# Rolling ES Severity Backtest (SPY): Same-day vs OOS (1-step shift)

**Updated:** 2026-03-16  
**Goal:**
1) Compute **rolling Historical VaR** + **rolling Historical ES** on SPY returns  
2) Compare **same-day** vs **out-of-sample (OOS, 1-step shift)** severity diagnostics  
3) Produce a compact ES “backtest-style” report for reporting  

## Conventions
- Log return (used as PnL): $$ r_t = \log(P_t) - \log(P_{t-1}) $$
- Loss: $$ L_t = -r_t $$
- VaR / ES are reported as **positive loss magnitudes**

## 1) Import & Data: SPY price → log returns → loss

We load SPY daily prices, convert to log returns (PnL), then define losses by the project convention:
- Log return (PnL): $$ r_t = \log(P_t) - \log(P_{t-1}) $$
- Loss: $$ L_t = -r_t $$

In [2]:
import numpy as np
import pandas as pd

from riskmetrics.var import rolling_historical_var
from riskmetrics.es import rolling_historical_es
from riskmetrics.backtest import (
    backtest_report,
    backtest_report_oos,
    es_backtest_report,
    es_backtest_report_oos,
)

df = pd.read_csv("../data/price_SPY.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

price = df["price"].astype(float)
pnl = np.log(price).diff().dropna()   # log returns as PnL
loss = -pnl                           # loss convention

print("n_prices:", len(price), "n_returns:", len(pnl))
loss.head()

n_prices: 1256 n_returns: 1255


date
2021-03-09   -0.014177
2021-03-10   -0.006205
2021-03-11   -0.010088
2021-03-12   -0.001346
2021-03-15   -0.005946
Name: price, dtype: float64

## 2) Rolling VaR & Rolling ES (one $$(\alpha,\ \mathrm{window})$$)

We compute rolling historical VaR and ES using the same rolling window:
- Rolling VaR:

$$ \mathrm{VaR}{\alpha,t} = q{\alpha}!\left(L_{t-w+1},\dots,L_t\right) $$


- Rolling ES:

$$ \mathrm{ES}{\alpha,t} = \mathbb{E}!\left[L \mid L \ge \mathrm{VaR}{\alpha,t}\right] \quad \text{(estimated within the window)} $$

In [3]:
# knobs
alpha = 0.99
window = 250

rvar = rolling_historical_var(pnl, window=window, alpha=alpha)
res  = rolling_historical_es(pnl, window=window, alpha=alpha)

print("rvar tail:")
print(rvar.dropna().tail(3))
print("\nres tail:")
print(res.dropna().tail(3))

rvar tail:
date
2026-03-04    0.036278
2026-03-05    0.036278
2026-03-06    0.036278
Name: price, dtype: float64

res tail:
date
2026-03-04    0.05189
2026-03-05    0.05189
2026-03-06    0.05189
Name: price, dtype: float64


## 3) VaR coverage report (same-day vs OOS)

We replicate the VaR coverage report first, so ES severity diagnostics are aligned with the same VaR backtest setting.
- Same-day:

$$ I_t^{\mathrm{same}} = \mathbf{1}{{,L_t > \mathrm{VaR}{\alpha,t},}} $$

- OOS (shift1):

$$ I_t^{\mathrm{oos}} = \mathbf{1}{{,L_t > \mathrm{VaR}{\alpha,t-1},}} $$

In [4]:
rep_same = backtest_report(loss, rvar, alpha=alpha)
rep_oos  = backtest_report_oos(loss, rvar, alpha=alpha)

print("=== VaR report: SAME-DAY ===")
print(rep_same)
print("\n=== VaR report: OOS (shift1) ===")
print(rep_oos)

=== VaR report: SAME-DAY ===
{'n': 1006, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.018886679920477135, 'lr_pof': 6.363619565091511, 'p_value': 0.011648368066473513}

=== VaR report: OOS (shift1) ===
{'n': 1005, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.01890547263681592, 'lr_pof': 6.38167266408999, 'p_value': 0.01153047141963337}


## 4) ES severity report (same-day vs OOS)

ES is not usually “backtested” by a simple coverage count.
Instead, we summarize severity diagnostics conditional on VaR violations:
- Mean realized loss on violation days: 

$$ \mathbb{E}!\left[L_t \mid L_t > \mathrm{VaR}\right] $$

- Mean ES value on violation days:

$$ \mathbb{E}!\left[\mathrm{ES}_t \mid L_t > \mathrm{VaR}\right] $$
- Mean excess over VaR:

$$ \mathbb{E}!\left[L_t - \mathrm{VaR} \mid L_t > \mathrm{VaR}\right] $$

- Mean excess over ES: 

$$ \mathbb{E}!\left[L_t - \mathrm{ES} \mid L_t > \mathrm{VaR}\right] $$


### Interpretation:
- If $\mathbb{E}[L_t - \mathrm{VaR}]$ is large, violations are not only frequent but also severe.
- If $\mathbb{E}[L_t - \mathrm{ES}]$ is large, even ES underestimates realized tail losses on violation days.

In [5]:
es_same = es_backtest_report(loss, rvar, res, alpha=alpha)
es_oos  = es_backtest_report_oos(loss, rvar, res, alpha=alpha)

print("=== ES report: SAME-DAY ===")
print(es_same)
print("\n=== ES report: OOS (shift1) ===")
print(es_oos)

=== ES report: SAME-DAY ===
{'n': 1006, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.018886679920477135, 'mean_loss_given_violation': 0.033620555240703315, 'mean_es_given_violation': 0.03146677972288038, 'mean_excess_over_var': 0.007373500149360478, 'mean_excess_over_es': 0.0021537755178229396}

=== ES report: OOS (shift1) ===
{'n': 1005, 'x': 19, 'expected_rate': 0.010000000000000009, 'observed_rate': 0.01890547263681592, 'mean_loss_given_violation': 0.033620555240703315, 'mean_es_given_violation': 0.028587321153933178, 'mean_excess_over_var': 0.009437297166065048, 'mean_excess_over_es': 0.005033234086770134}


## 5) Diagnostic table (sanity check)

We create an aligned table to sanity-check that:
- indices match (no hidden alignment bugs)
- violations correspond exactly to $L_t > \mathrm{VaR}$
- ES is defined on the same aligned timeline

We include both:
- same-day violations using $\mathrm{VaR}_{\alpha,t}$
- OOS violations using $\mathrm{VaR}_{\alpha,t-1}$

In [6]:
aligned = pd.concat(
    [loss.rename("loss"), rvar.rename("VaR"), res.rename("ES")],
    axis=1
).dropna()

aligned["violation_same"] = (aligned["loss"] > aligned["VaR"]).astype(int)

# OOS uses VaR_{t-1} (shift)
aligned["VaR_prev"] = aligned["VaR"].shift(1)
aligned["violation_oos"] = (aligned["loss"] > aligned["VaR_prev"]).astype(int)

aligned.tail(10)

,loss,VaR,ES,violation_same,VaR_prev,violation_oos
date,,,,,,
2026-02-23,0.010264,0.036278,0.05189,0,0.036278,0
2026-02-24,-0.007242,0.036278,0.05189,0,0.036278,0
2026-02-25,-0.008403,0.036278,0.05189,0,0.036278,0
2026-02-26,0.005570,0.036278,0.05189,0,0.036278,0
2026-02-27,0.004814,0.036278,0.05189,0,0.036278,0
2026-03-02,-0.000568,0.036278,0.05189,0,0.036278,0
2026-03-03,0.008853,0.036278,0.05189,0,0.036278,0
2026-03-04,-0.007031,0.036278,0.05189,0,0.036278,0
2026-03-05,0.005591,0.036278,0.05189,0,0.036278,0


## 6) Summary

### Window length
- Short windows often have noisier tail estimates → observed violation rate can exceed expected (under-coverage).
- Longer windows stabilize quantiles and can improve coverage.

### Confidence level
- $\alpha = 0.99$ is harder: extreme-tail estimation is sensitive to heavy tails / volatility clustering.

### Same-day vs OOS
- OOS uses one fewer effective observation (shift).
- Differences can be small for large windows, but OOS is preferred for leakage-safe evaluation.

### Practical takeaway

Rolling ES is best used as a tail severity diagnostic (how bad violations are), rather than a simple “coverage count” like VaR.